In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi*(rand(1)[1]/4+0.875)
ω_b = 2*ω_a

φ_a = 0.11*(rand(1)[1]/5+0.9)
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204*(rand(1)[1]/5+0.9)
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5*(rand(1)[1]/10+0.95)
r=sqrt(Z_b/Z_a)

##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

$$H_\text{min} = \omega_a a^*a + \omega_b b^*b +\varepsilon_d\cos(\omega_b t)(b+b^*)\\
        +E_J\sin(\varepsilon_p)\varphi_a^3\biggl\{(a+rb)^2(a^*+rb^*) - a^*r^2b^2\\
        -\varphi_b^2bb^*\bigl[a^*a(a+rb) + r^3b^2b^*/6\bigl] + H.c.\biggl\}$$
$r=\varphi_a/\varphi_b$

In [ ]:
function test_equation_minimum_bis!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    r=sqrt(Z_b/Z_a)
    φ_a = sqrt(Z_a/(2*ϕ_quantum^2))
    φ_b = sqrt(Z_b/(2*ϕ_quantum^2))

    sin_pump = E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)*φ_a^3

    #diff_a =(a+r*b)*(a+2*conj(a)+r*b+2*r*conj(b))-r^2*b^2-φ_b^2*a*b*conj(b)*(a+2*conj(a)+r*b+r*conj(b))
    diff_a = a*(a+conj(a))+a*conj(a)+2*r*a*(b+conj(b))+2*r*conj(a)*b+2*r^2*b*conj(b)-φ_b^2*a*b*conj(b)*(a+2*conj(a)+r*b+r*conj(b))

    #diff_b= (a+r*b)*(a+2*conj(a)+r*b+2*r*conj(b))+r^2*conj(b)^2-2*a*r*conj(b)-φ_b^2*b/r*(conj(a)*a*(a+conj(a)+r*b+2*r*conj(b))+r^3*b*conj(b)*(conj(b)/2+b/3))
    #diff_b= (a+r*b)*(a+2*conj(a)+r*b+2*r*conj(b))-2*a*r*conj(b)-φ_b^2*b/r*(conj(a)*a*(a+conj(a)+r*b+2*r*conj(b))+r^3*b*conj(b)*(conj(b)/2+b/3))
    diff_b= 2*a*conj(a)+a^2+2*r*(a+conj(a))*b+r^2*b*(b+conj(b))+r^2*b*conj(b)-φ_b^2*b/r*(conj(a)*a*(a+conj(a)+r*b+2*r*conj(b))+r^3*b*conj(b)*(conj(b)/2+b/3))
        
    du[1] = -1im*(ω_a*a + sin_pump*diff_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + r*sin_pump*diff_b) - p.environment_prepared.κ_b * b
    return nothing
end

# Check for the Mean Field

In [ ]:
ϵ_0 = 0.8#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=(10.12*(rand(1)[1]/5+0.9))*g2

t_max=20/κ_b
re_a_0, im_a_0 = 0, 0
a_0 = re_a_0 + 1im* im_a_0
b_0 = 0

system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

a, b, t_list_CL = simulator(t_max, system, drive, environment, equation = test_equation_minimum_bis!)
a_rot = a.*exp.(1im*ω_a*t_list_CL);

In [ ]:
plot(real.(a_rot), imag.(a_rot))

In [ ]:
excitation_sharing_list = [(1,2,0,0),(1,1,1,0),(0,2,1,0),(0,1,1,1),(0,0,1,2),(2,1,1,1),(1,1,2,1),(0,0,2,3)]

coef_dict = Dict(
    (i,j,k,l) => φ_a^(i+j+k+l)*r^(k+l) / prod(factorial.((i,j,k,l)))
    for (i,j,k,l) in excitation_sharing_list
)


function excitation_to_monomial(excitation) 
    pow_a, pow_a_dag, pow_b, pow_b_dag = excitation
    string_a, string_a_dag, string_b, string_b_dag = "", "", "", "" 
    if pow_a != 0 
        string_a = latexstring("a^", "{$pow_a}") 
    end 
    if pow_a_dag != 0 
        string_a_dag = latexstring("(a^\\dagger)^", "{$pow_a_dag}") 
    end 
    if pow_b != 0
        string_b = latexstring("b^", "{$pow_b}") 
    end 
    if pow_b_dag != 0
        string_b_dag = latexstring("(b^\\dagger)^", "{$pow_b_dag}") 
    end 
    return string_a*string_a_dag*string_b*string_b_dag end


ϵ_0 = 0.9#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2
r=sqrt(Z_b/Z_a)
t_max=20/κ_b
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a_0 = rand(Float32)/10 + 1im* rand(Float32)/10
b_0 = rand(Float32)/10 + 1im* rand(Float32)/10

system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0, b_0)
a, b, t_list = simulator(t_max, system, drive, environment)

In [ ]:
monomial_list = Dict{NTuple{4,Int64},Vector{ComplexF32}}()
for excitation_sharing in excitation_sharing_list
    pow_a, pow_a_dag, pow_b, pow_b_dag = excitation_sharing
    coef = coef_dict[excitation_sharing]
    expr = @. a^pow_a * conj(a)^pow_a_dag * (r*b)^pow_b * (r*conj(b))^pow_b_dag
    monomial_list[excitation_sharing] = expr#coef*expr
end

In [ ]:
start = 1
step = 50
stop = length(t_list)
for excitation in excitation_sharing_list
    p0 = plot(real.(monomial_list[(excitation)][start:step:stop]), imag.(monomial_list[(excitation)][start:step:stop]), label = excitation_to_monomial(excitation))
    display(p0)
end

In [ ]:
start = 1
step = 50
stop = length(t_list)
for excitation in excitation_sharing_list
    p0 = plot(real.(monomial_list[(excitation)][start:step:stop]), imag.(monomial_list[(excitation)][start:step:stop]), label = excitation_to_monomial(excitation))
    display(p0)
end

In [ ]:
pairs = collect(zip(.-max_monomial_list, excitation_sharing_list))
pairs_sorted = sort(pairs; by = first)

# Recover sorted monomials
excitation_sharing_list_sorted = last.(pairs_sorted)
max_sorted = .-first.(pairs_sorted);

monomial_list = excitation_to_monomial.(excitation_sharing_list_sorted);

plot(
    monomial_list,
    max_sorted,
    xrotation = 90,
    yaxis=:log,
    bottommargin = 10mm,
    label=""
)

# Poincaré section

In [ ]:
ϵ_0_list = range(start = 0.50, stop = 0.702, length = 10);


t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(sqrt(2)*ω_a)*3
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,25)
Final_all = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = test_equation_minimum_bis!)

            a_rot = a.*exp.(1im*ω_a*t_list)

            Final_mode[(re_i,im_j)] = a_rot
        end
    end
    Final_all[ϵ_0] = Final_mode
end

In [ ]:
min_re = 0
max_re = 0
min_im = 0
max_im = 0
for ϵ_0 in ϵ_0_list
    Final_mode = Final_all[ϵ_0]
    for (re_i, im_j) in keys(Final_mode)
        array = Final_mode[(re_i, im_j)]
        if minimum(real.(array))< min_re 
            min_re = minimum(real.(array))
        end
        if maximum(real.(array))> max_re 
            max_re = maximum(real.(array))
        end
        if minimum(imag.(array))< min_im
            min_im = minimum(imag.(array))
        end
        if maximum(imag.(array))> max_im 
            max_im = maximum(imag.(array))
        end
    end
end
min_re, max_re, min_im, max_im

In [ ]:
for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b
    Final_mode = Final_all[ϵ_0]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=1, label="", color=:black)
    end

    # Add labels and title
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          xlims=(min_re, max_re),
          ylims=(min_im, max_im),
          top_margin=5mm)

    display(p0)
end
